# Machine Learning for Business Applications: Loan Default Prediction
### Project: German Credit Risk Assessment Tool
**Course:** MLBA | **Objective:** Build an end-to-end classification pipeline and export model for Streamlit web deployment.

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Load the dataset
df = pd.read_csv('../data/german_credit_data.csv')
print('Dataset Shape:', df.shape)
df.head()

## 2. Data Understanding & Target Preparation

In [ ]:
# Check info and missing values
df.info()

# Class distribution
print('\nClass counts:')
print(df['class'].value_counts())

# Map target: 1 = Bad (Defaulter / High Risk), 0 = Good (Repayer / Low Risk)
df['target'] = df['class'].map({'bad': 1, 'good': 0})

## 3. Exploratory Data Analysis (EDA) & Business Insights
We analyze key drivers of credit risk: Checking balance, loan duration, and credit amount.

In [ ]:
# Chart 1: Default Risk Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='class', data=df, hue='class', palette=['#2ecc71', '#e74c3c'], legend=False)
plt.title('Figure 1: Credit Risk Distribution (Good vs. Bad Loans)', fontweight='bold')
plt.xlabel('Credit Risk')
plt.ylabel('Number of Applicants')
plt.show()

In [ ]:
# Chart 2: Checking Account Status vs Default Rate
plt.figure(figsize=(7, 4.5))
chk_risk = df.groupby('checking_status')['target'].mean().reset_index()
sns.barplot(x='checking_status', y='target', data=chk_risk, hue='checking_status', palette='Blues_r', legend=False)
plt.title('Figure 2: Default Rate by Checking Account Balance', fontweight='bold')
plt.ylabel('Default Rate (% Defaulters)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.show()

In [ ]:
# Chart 3: Loan Duration vs Risk
plt.figure(figsize=(6, 4.5))
sns.boxplot(x='class', y='duration', data=df, hue='class', palette=['#2ecc71', '#e74c3c'], legend=False)
plt.title('Figure 3: Loan Duration (Months) by Risk Category', fontweight='bold')
plt.xlabel('Credit Risk')
plt.ylabel('Duration in Months')
plt.show()

## 4. Feature Selection & Train/Test Split

In [ ]:
# Selected core features for application and modeling
selected_features = [
    'checking_status', 'duration', 'credit_amount', 'savings_status', 
    'employment', 'age', 'housing', 'purpose'
]

X = df[selected_features]
y = df['target']

num_cols = ['duration', 'credit_amount', 'age']
cat_cols = ['checking_status', 'savings_status', 'employment', 'housing', 'purpose']

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training samples:', len(X_train), '| Test samples:', len(X_test))

## 5. Build Preprocessing & Modeling Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced'))
])

pipeline.fit(X_train, y_train)
print('Pipeline fitted successfully.')

## 6. Model Evaluation & Business Interpretation

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('=== Classification Metrics ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'F1-Score:  {f1_score(y_test, y_pred):.3f}')

print('\n=== Confusion Matrix ===')
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Good', 'Predicted Bad'], yticklabels=['Actual Good', 'Actual Bad'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix on Test Data')
plt.show()

## 7. Export Trained Model for Streamlit Deployment
We serialize the complete pipeline into model.pkl using joblib. This packages both the preprocessing scalers/encoders and the Random Forest model together.

In [ ]:
# Save the model
joblib.dump(pipeline, '../model.pkl')
print('Model exported to ../model.pkl successfully!')